# M11. 데이터 결합 (merge) ⭐

> 📌 **언제 필요한가**  
> 여러 데이터를 하나로 합쳐서 분석할 때.  
> 예: "시도별 교사 수" + "시도별 진학률" → 두 데이터를 합쳐서 비교

## 이 모듈에서 배울 것

- `pd.merge` 기본 사용법
- 4가지 join 방식 (inner, outer, left, right)
- **🔥 merge 디버깅** — 결과가 이상할 때 무엇을 의심하나
- 작년 선배(김서윤)가 만난 진짜 함정 사례

---

## 📥 데이터 준비

> 이 모듈은 아래 파일이 필요해요. **저장소에는 동봉돼 있지 않으니** 먼저 받아서 두세요.
> 받는 곳 링크를 누르면 바로 받으러 갈 수 있어요.

- `교육부_시도별 진로전담교사 배치현황_20241231.csv` — 인코딩 `cp949` — [공공데이터포털에서 받기](https://www.data.go.kr/data/15097012/fileData.do)
- `한국교육개발원_시도 시군구별 졸업자 진학자 진학률_20240401.csv` — 인코딩 `cp949` — [공공데이터포털에서 받기](https://www.data.go.kr/data/15053808/fileData.do)
- `교육부_지방교육재정정보_순계_2024.csv` — 인코딩 `cp949` — [공공데이터포털](https://www.data.go.kr/data/15052877/fileData.do) → [지방교육재정알리미](https://www.eduinfo.go.kr/portal/open/openData/dataSetPage.do) (예산통합공시 > 재정규모 > 통합재정 > 순계)

두는 곳 — **로컬 Jupyter**: 이 노트북과 같은 폴더 / **Colab**: `/content/`에 업로드.  
컬럼 설명·함정 등 자세한 내용은 [`data/README.md`](data/README.md) 참고.

---

## 1. merge 기본 — 두 데이터 합치기


In [ ]:
import pandas as pd

# 두 데이터 불러오기
counselor = pd.read_csv('data/교육부_시도별 진로전담교사 배치현황_20241231.csv', encoding='cp949')
graduate = pd.read_csv('data/한국교육개발원 시도 시군구별 졸업자 진학자 진학률_20240401.csv', encoding='cp949')

print("진로전담교사:", counselor.shape)
print(counselor.head(3))
print()
print("졸업자/진학자:", graduate.shape)
print(graduate.head(3))


진로전담교사 데이터는 **시도별** (17개 행), 졸업자 데이터는 **시군구별** (수백 개 행).

두 데이터를 합치려면 먼저 졸업자 데이터를 시도 단위로 집계해야 해요.


In [ ]:
# 졸업자 데이터를 시도별로 집계
graduate_by_sido = graduate.groupby('시도')[['졸업자', '진학자']].sum().reset_index()
print(graduate_by_sido.head())


In [ ]:
# 이제 두 시도별 데이터를 merge
merged = pd.merge(counselor, graduate_by_sido, on='시도')
print(f"merge 결과 shape: {merged.shape}")
merged.head()


**🎉 성공!** `on='시도'`로 두 데이터의 시도 컬럼을 키로 합쳤어요.


## 2. merge 4가지 방식

| 방식 | 결과 |
|---|---|
| `how='inner'` (기본) | 양쪽에 다 있는 키만 |
| `how='outer'` | 한쪽에라도 있는 모든 키 (없는 값은 NaN) |
| `how='left'` | 왼쪽 데이터 기준 (오른쪽에 없으면 NaN) |
| `how='right'` | 오른쪽 데이터 기준 |


In [ ]:
# 예시 데이터로 시연
import pandas as pd

A = pd.DataFrame({'키': ['a', 'b', 'c'], '값A': [1, 2, 3]})
B = pd.DataFrame({'키': ['b', 'c', 'd'], '값B': [20, 30, 40]})

print("A:")
print(A)
print("\nB:")
print(B)


In [ ]:
# inner (기본)
print("inner — 양쪽에 다 있는 'b', 'c'만:")
print(pd.merge(A, B, on='키', how='inner'))


In [ ]:
# outer
print("outer — 모든 키 ('a', 'b', 'c', 'd'):")
print(pd.merge(A, B, on='키', how='outer'))


In [ ]:
# left
print("left — A 기준 ('a', 'b', 'c'):")
print(pd.merge(A, B, on='키', how='left'))


## 3. 🔥 merge 디버깅 — 결과가 이상할 때

**작년 선배(김서윤)가 진짜로 만난 사례.**

세 번째 데이터(교육예산)를 merge하려고 했는데...


In [ ]:
budget = pd.read_csv('data/교육부_지방교육재정정보_순계_2024.csv', encoding='cp949')
budget = budget.dropna(axis=1)  # 빈 컬럼 제거
budget.columns = budget.columns.str.strip()  # 컬럼명 앞뒤 공백 제거 (받은 파일은 ' 지역구분'처럼 앞 공백)
budget = budget[['지역구분', '금액(원)']]
budget.columns = ['시도', '예산']
budget.head()


In [ ]:
# 보기엔 정상. 합쳐보자.
result = pd.merge(merged, budget, on='시도')
print(f"shape: {result.shape}")
result


**❌ 어, 결과가 비었어요!** 양쪽 데이터에 시도가 다 있는데 왜?

이럴 때 의심해야 할 #1 원인: **키 컬럼의 값이 정확히 같지 않을 가능성**.

### 디버깅 — unique() 비교


In [ ]:
# 양쪽 키 컬럼의 unique 값 비교
print("merged의 시도:")
print(merged['시도'].unique())
print()
print("budget의 시도:")
print(budget['시도'].unique())


**보이세요?** 눈으로는 똑같아 보이지만, `repr()`로 보면 다름:


In [ ]:
# repr로 보면 진짜 차이가 보임
print("merged의 첫 시도값:", repr(merged['시도'].iloc[0]))
print("budget의 첫 시도값:", repr(budget['시도'].iloc[0]))


**범인 발견!** budget 쪽 시도명 끝에 **공백** 한 칸이 붙어 있음.  
`'서울특별시'` ≠ `'서울특별시 '` 이므로 merge 안 됨.

### 해결 — str.strip()


In [ ]:
# 공백 제거
budget['시도'] = budget['시도'].str.strip()

# 다시 merge
result = pd.merge(merged, budget, on='시도')
print(f"shape: {result.shape}")
result


**🎉 해결!** 이제 17개 시도 다 잘 합쳐졌어요.

> 💡 **이게 작년 선배가 만난 진짜 함정이에요.**  
> 공공데이터에서 흔하니까 머릿속에 새겨두세요: **merge 안 되면 `unique()` 비교 + `str.strip()`**


## 4. merge 디버깅 체크리스트

merge 결과가 이상하면 이 순서로 확인:

```python
# 1. 양쪽 키 컬럼의 unique 비교
print(A['키'].unique())
print(B['키'].unique())

# 2. repr로 공백/특수문자 확인  
print(repr(A['키'].iloc[0]))
print(repr(B['키'].iloc[0]))

# 3. 양쪽 다 strip
A['키'] = A['키'].str.strip()
B['키'] = B['키'].str.strip()

# 4. 그래도 안 되면 표기 통일
mapping = {'서울': '서울특별시', ...}
B['키'] = B['키'].replace(mapping)
```

흔한 원인:
- ✅ **공백 차이** (`str.strip()`)
- 대소문자 차이 (`str.lower()`)
- 띄어쓰기 차이 (`str.replace(' ', '')`)
- 공식 명칭 차이 (`replace` 매핑)


## 5. 본인 데이터에 적용해보기 ✏️


In [ ]:
# 본인 데이터 두 개 merge
# A = pd.read_csv('데이터1.csv', encoding='cp949')
# B = pd.read_csv('데이터2.csv', encoding='cp949')
# 
# # Step 1: 공통 키 컬럼 정하기 (양쪽에 같은 의미의 컬럼)
# key = '시도'  # 본인 데이터에 맞게
# 
# # Step 2: 일단 시도해보기
# result = pd.merge(A, B, on=key)
# print(f"shape: {result.shape}")
# 
# # Step 3: 결과가 이상하면 unique 비교
# if result.shape[0] < min(A.shape[0], B.shape[0]) * 0.8:
#     print("⚠️ merge가 잘 안 된 것 같음. 디버깅 시작...")
#     print(A[key].unique()[:5])
#     print(B[key].unique()[:5])


## 6. ⚠️ 함정 / 주의사항

### 6.1 키 컬럼명이 다른 경우
양쪽 데이터에서 키 컬럼 이름이 다르면 `left_on`, `right_on` 사용:
```python
pd.merge(A, B, left_on='시도', right_on='지역')
```

### 6.2 키가 여러 개
```python
pd.merge(A, B, on=['시도', '연도'])  # 두 컬럼 모두 일치할 때만
```

### 6.3 중복 키
양쪽에 같은 키가 여러 개면 곱(cartesian product)이 되어서 행 수 폭증.
```python
A.duplicated(subset='시도').sum()  # 중복 확인
```

### 6.4 merge 후 같은 이름의 컬럼
이름 충돌 시 자동으로 `_x`, `_y` 붙임:
```python
pd.merge(A, B, on='키', suffixes=('_A', '_B'))  # 명시적으로 지정
```


## 7. 📚 더 알아보기

- `pd.concat([A, B])` — 세로로 이어붙이기 (같은 컬럼)
- `pd.concat([A, B], axis=1)` — 가로로 이어붙이기 (같은 인덱스)
- `df.join(other)` — 인덱스 기준 결합 (간편한 merge)
- `df.combine_first(other)` — 결측 채우면서 결합
